In [ ]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Environment ready")
print("TensorFlow version:", tf.__version__)

In [ ]:
DATASET_PATH = r"train_with_images.csv" 
df = pd.read_csv(DATASET_PATH)

print("Loaded dataframe shape:", df.shape)
print("Columns:", df.columns.tolist())


print("Dataset loaded & verified")


## Target engineering

In [ ]:
assert (df["price"] >= 0).all(), "Negative prices found"

# Log-transform target
df["log_price"] = np.log1p(df["price"])

df = df.dropna(subset=["image_path", "log_price"]).reset_index(drop=True)

assert np.isfinite(df["log_price"]).all()
assert df["image_path"].notna().all()

print("Cell 3 executed:")
print("Final dataframe shape:", df.shape)
print(df[["price", "log_price"]].head())


## Reproducibility

In [ ]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


## Define tabular feature columns

In [ ]:
TABULAR_COLUMNS = [
    "bedrooms", "bathrooms", "sqft_living", "sqft_lot",
    "floors", "waterfront", "view", "condition", "grade",
    "sqft_above", "sqft_basement",
    "yr_built",
    "lat", "long",
    "sqft_living15", "sqft_lot15"
]

print("Cell 4 executed: Tabular columns verified")
print("Number of tabular features:", len(TABULAR_COLUMNS))


## Train / validation split

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED
)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)


## Scale tabular features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_tab = scaler.fit_transform(df_train[TABULAR_COLUMNS])
X_val_tab   = scaler.transform(df_val[TABULAR_COLUMNS])

# Targets (log-price)
y_train = df_train["log_price"].values.astype("float32")
y_val   = df_val["log_price"].values.astype("float32")

assert np.isfinite(X_train_tab).all()
assert np.isfinite(X_val_tab).all()
assert np.isfinite(y_train).all()
assert np.isfinite(y_val).all()

print("Cell 6 executed:")
print("X_train_tab shape:", X_train_tab.shape)
print("X_val_tab shape:", X_val_tab.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)


## Build tabular-only baseline model

In [ ]:
from tensorflow.keras import layers, models

tabular_input = tf.keras.Input(
    shape=(X_train_tab.shape[1],),
    name="tabular_input",
    dtype="float32"
)

x = layers.Dense(128, activation="relu")(tabular_input)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dense(32, activation="relu")(x)

output = layers.Dense(1, name="log_price")(x)

tabular_model = models.Model(
    inputs=tabular_input,
    outputs=output
)

tabular_model.summary()

## Compile baseline model

In [ ]:
from tensorflow.keras.optimizers import Adam

tabular_model.compile(
    optimizer=Adam(
        learning_rate=1e-4,   # conservative for stability
        clipnorm=1.0          # prevents gradient explosion
    ),
    loss=tf.keras.losses.Huber(delta=1.0)  
)

print("Cell 8 executed: Baseline model compiled safely")

## Train tabular-only baseline model

In [ ]:
history_tabular = tabular_model.fit(
    X_train_tab,
    y_train,
    validation_data=(X_val_tab, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True
        )
    ],
    verbose=1
)
